# 06 · OCR piloto en Jupyter (sin instalar Tesseract)

Este notebook compara **12 páginas** de los seis PDF con **RapidOCR** (modelo de reconocimiento latino) y el texto extraído anteriormente con PyMuPDF. Se ejecuta **una sola vez**, desde `notebooks/`, y crea `data/output/etapa6.zip` automáticamente. Usa `etapa5.zip` y `etapa4.zip` sin cambiar rutas.

**Necesita los PDF ya descargados en `data/pdf/`.** No vuelve a descargarlos. La primera ejecución puede instalar paquetes Python y descargar el modelo de reconocimiento; requiere internet. La elección del modelo latino no garantiza exactitud para impresos coloniales, quechua o aymara.

**Salidas:** `etapa6.zip` (informe y extractos breves); `data/output/etapa6/revision_ocr.html` (comparación visual local), `data/output/etapa6/vistas/` (imágenes locales de páginas), `data/output/etapa6/transcripciones_locales.csv` (texto OCR completo *solo local*). El ZIP no contiene PDF ni páginas escaneadas ni transcripciones íntegras. **Ningún texto queda automáticamente aprobado para RAG.**

In [1]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone
import io, json, hashlib, sys, subprocess, importlib.util, html, statistics, re
import pandas as pd

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name.lower() == 'notebooks' else CWD
DATA = ROOT / 'data'
assert DATA.is_dir(), f'No encuentro data/ desde {CWD}. Abre Jupyter dentro de este proyecto.'
PDF_DIR = DATA / 'pdf'
OUT = DATA / 'output' / 'etapa6'
OUT.mkdir(parents=True, exist_ok=True)
VISTAS = OUT / 'vistas'
VISTAS.mkdir(parents=True, exist_ok=True)

def localizar_zip(nombre):
    for p in (DATA / 'output' / nombre, DATA / 'input' / nombre):
        if p.is_file(): return p
    raise FileNotFoundError(f'Falta {nombre} en data/output/ o data/input/')

ZIP4 = localizar_zip('etapa4.zip')
ZIP5 = localizar_zip('etapa5.zip')
with ZipFile(ZIP5) as z:
    muestra = pd.read_csv(io.BytesIO(z.read('muestra_revision_manual.csv'))).fillna('')
    informe5 = json.loads(z.read('informe_etapa5.json').decode('utf-8'))
with ZipFile(ZIP4) as z:
    originales = pd.read_csv(io.BytesIO(z.read('paginas_texto.csv'))).fillna({'texto':''})
assert len(muestra) == 24 and muestra.mcid.nunique() == 6, 'La muestra no coincide con la auditoría anterior'
assert hashlib.sha256(ZIP4.read_bytes()).hexdigest() == informe5['etapa4_sha256'], 'etapa4.zip no coincide con el usado en etapa 5'

criterios = ['texto_con_senales_de_ruido','texto_pendiente_validacion']
seleccion = pd.concat([muestra[muestra.criterio_muestreo == c] for c in criterios], ignore_index=True)
assert len(seleccion) == 12, f'Se esperaban 12 páginas: {len(seleccion)}'
assert seleccion[['mcid','pagina_pdf']].drop_duplicates().shape[0] == 12
faltan = sorted(set(seleccion.mcid) - {p.stem for p in PDF_DIR.glob('*.pdf')})
print('Proyecto:', ROOT)
print('Muestra OCR:', len(seleccion), 'páginas de', seleccion.mcid.nunique(), 'documentos')
print('PDF locales faltantes:', faltan)
if faltan:
    raise FileNotFoundError('Conserva tus PDF originales en data/pdf/ (por ejemplo MC0014753.pdf). No vuelvas a descargar los PDF desde este notebook. Faltan: ' + ', '.join(faltan))

Proyecto: C:\Users\jvera\Documents\memoria_chilena
Muestra OCR: 12 páginas de 6 documentos
PDF locales faltantes: []


## Instalar dependencias desde Jupyter

Solo se instalan paquetes Python faltantes. No es necesario instalar Tesseract ni ejecutar archivos `.bat`. RapidOCR utiliza ONNX Runtime en CPU. Si la instalación falla, la celda mostrará el error específico; evita continuar a ciegas.

In [2]:
paquetes = {'fitz':'PyMuPDF>=1.24,<2','rapidocr':'rapidocr>=3.10,<4','onnxruntime':'onnxruntime>=1.18,<2','PIL':'Pillow>=10,<13','numpy':'numpy>=1.24,<3'}
faltantes=[pip for modulo,pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    print('Instalando:', ', '.join(faltantes))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *faltantes])
else:
    print('Dependencias OCR disponibles.')
import fitz
import numpy as np
from PIL import Image
from rapidocr import RapidOCR, EngineType, ModelType, OCRVersion

# PP-OCRv5 reconoce alfabetos latinos. No presupone que sea adecuado para tipografía histórica.
config_ocr = {
    'Rec.engine_type': EngineType.ONNXRUNTIME,
    'Rec.lang_type': 'latin',
    'Rec.ocr_version': OCRVersion.PPOCRV5,
    'Rec.model_type': ModelType.MOBILE,
}
print('Inicializando RapidOCR para texto latino; puede requerir descargar el modelo...')
motor = RapidOCR(params=config_ocr)
print('Modelo inicializado.')

Instalando: rapidocr>=3.10,<4


C:\Users\jvera\anaconda3\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Inicializando RapidOCR para texto latino; puede requerir descargar el modelo...
Modelo inicializado.


## Piloto de 12 páginas

Una página marcada con señales de ruido y otra pendiente de validación por documento. Se conserva el texto nativo, las transcripciones completas en el equipo, el número de página del PDF y la huella digital del PDF. No se calculan CER/WER sin una transcripción humana de referencia.

In [3]:
mapa_original = {(str(r.mcid),int(r.pagina_pdf)):str(r.texto or '') for r in originales.itertuples(index=False)}
filas=[]
transcripciones=[]
for mcid, grupo in seleccion.groupby('mcid',sort=True):
    pdf = PDF_DIR / f'{mcid}.pdf'
    dig=hashlib.sha256()
    with open(pdf,'rb') as fh:
        for bloque in iter(lambda: fh.read(1024*1024), b''): dig.update(bloque)
    sha=dig.hexdigest()
    with fitz.open(pdf) as doc:
        for r in grupo.sort_values('pagina_pdf').itertuples(index=False):
            pagina=int(r.pagina_pdf)
            item={'mcid':mcid, 'pagina_pdf':pagina,'criterio':r.criterio_muestreo,
                  'ficha':r.ficha, 'fuente_pdf':r.fuente_pdf,'sha256_pdf':sha,
                  'caracteres_nativo':len(mapa_original.get((mcid,pagina),'')),
                  'caracteres_ocr':0,'confianza_media_modelo':None,'estado':'','error':''}
            try:
                if not (1 <= pagina <= len(doc)): raise ValueError('Número de página fuera del PDF')
                page=doc[pagina-1]
                escala=min(2.5, 2400/max(page.rect.width,page.rect.height))
                escala=max(1.0, escala)
                pix=page.get_pixmap(matrix=fitz.Matrix(escala,escala), alpha=False)
                ruta_vista=VISTAS / f'{mcid}_p{pagina:04d}.png'
                pix.save(str(ruta_vista))
                imagen=Image.open(io.BytesIO(pix.tobytes('png'))).convert('RGB')
                resultado=motor(imagen)
                salida_textos=getattr(resultado,'txts',None)
                salida_scores=getattr(resultado,'scores',None)
                textos=[str(t) for t in (salida_textos if salida_textos is not None else []) if str(t).strip()]
                valores=[float(x) for x in (salida_scores if salida_scores is not None else [])]
                texto_ocr='\n'.join(textos)
                item['caracteres_ocr']=len(texto_ocr)
                item['confianza_media_modelo']=round(statistics.mean(valores),3) if valores else None
                item['estado']='ocr_ejecutado' if texto_ocr else 'ocr_sin_texto'
                transcripciones.append({'mcid':mcid,'pagina_pdf':pagina,
                    'texto_nativo':mapa_original.get((mcid,pagina),''), 'texto_ocr':texto_ocr,
                    'criterio':r.criterio_muestreo,'sha256_pdf':sha})
            except Exception as e:
                item['estado']='error_ocr'
                item['error']=f'{type(e).__name__}: {e}'[:250]
            filas.append(item)
            print(mcid, 'página', pagina, '→', item['estado'], '| caracteres OCR:', item['caracteres_ocr'])

resultado_df=pd.DataFrame(filas)
textos_df=pd.DataFrame(transcripciones)
resultado_df.to_csv(OUT/'resumen_ocr_paginas.csv',index=False,encoding='utf-8-sig')
textos_df.to_csv(OUT/'transcripciones_locales.csv',index=False,encoding='utf-8-sig')
print('Páginas con OCR ejecutado:',int((resultado_df.estado=='ocr_ejecutado').sum()),'/12')
print('Errores:',int((resultado_df.estado=='error_ocr').sum()))

[INFO] 2026-11-09 13:44:52,512 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-09 13:44:52,816 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\jvera\anaconda3\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-11-09 13:44:52,818 [RapidOCR] main.py:63: Using C:\Users\jvera\anaconda3\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-11-09 13:44:53,897 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-09 13:44:53,905 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\jvera\anaconda3\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-09 13:44:53,908 [RapidOCR] main.py:63: Using C:\Users\jvera\anaconda3\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-09 13:44:54,182 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-09 13:44:54,187 [RapidOCR] download_file.py:68: Initiating download: https

MC0014753 página 104 → ocr_ejecutado | caracteres OCR: 656
MC0014753 página 140 → ocr_ejecutado | caracteres OCR: 1119
MC0014754 página 146 → ocr_ejecutado | caracteres OCR: 2495
MC0014754 página 156 → ocr_ejecutado | caracteres OCR: 2281
MC0014755 página 213 → ocr_ejecutado | caracteres OCR: 2631
MC0014755 página 295 → ocr_ejecutado | caracteres OCR: 2732
MC0014757 página 211 → ocr_ejecutado | caracteres OCR: 1529
MC0014757 página 311 → ocr_ejecutado | caracteres OCR: 1823
MC0014760 página 72 → ocr_ejecutado | caracteres OCR: 2461
MC0014760 página 83 → ocr_ejecutado | caracteres OCR: 2518
MC0014761 página 505 → ocr_ejecutado | caracteres OCR: 2277
MC0014761 página 554 → ocr_ejecutado | caracteres OCR: 2054
Páginas con OCR ejecutado: 12 /12
Errores: 0


## Comparar con las páginas (solo local)

Abre `data/output/etapa6/revision_ocr.html`. Allí se muestran, lado a lado, la imagen de la página, el texto nativo y la salida OCR. El archivo queda únicamente en tu computadora y **no se incluye en `etapa6.zip`**.

In [4]:
tramos=['<!doctype html><html lang="es"><meta charset="utf-8"><title>Revisión OCR - Memoria Chilena</title>',
'<style>body{font-family:Arial,sans-serif;max-width:1550px;margin:2rem auto;padding:0 1rem}section{border-top:2px solid #aaa;padding:2rem 0}.triple{display:grid;grid-template-columns:1fr 1fr 1fr;gap:1.2rem}img{max-width:100%;max-height:800px;border:1px solid #bbb}pre{white-space:pre-wrap;overflow-wrap:anywhere;max-height:800px;overflow:auto;background:#f7f7f7;padding:1rem}small{color:#555}@media(max-width:900px){.triple{grid-template-columns:1fr}}</style>',
'<h1>Comparación OCR piloto — solo revisión local</h1><p>La confianza del motor no equivale a exactitud histórica. Todos los textos permanecen pendientes de validación.</p>']
for r in resultado_df.itertuples(index=False):
    d=textos_df[(textos_df.mcid==r.mcid)&(textos_df.pagina_pdf==r.pagina_pdf)]
    orig=d.iloc[0]['texto_nativo'] if len(d) else ''
    ocr=d.iloc[0]['texto_ocr'] if len(d) else ''
    ruta=f'vistas/{r.mcid}_p{r.pagina_pdf:04d}.png'
    tramos.append(f'<section><h2>{html.escape(r.mcid)} · PDF página {r.pagina_pdf}</h2><small>{html.escape(r.criterio)} · {html.escape(r.estado)}</small>')
    tramos.append('<div class="triple"><div><h3>Imagen PDF</h3>'+ (f'<img src="{ruta}" alt="Página del documento">' if (OUT/ruta).exists() else '<p>Imagen no disponible</p>') +'</div>')
    tramos.append('<div><h3>Texto nativo</h3><pre>'+html.escape(str(orig))+'</pre></div><div><h3>RapidOCR</h3><pre>'+html.escape(str(ocr))+'</pre></div></div></section>')
tramos.append('</html>')
(OUT/'revision_ocr.html').write_text('\n'.join(tramos),encoding='utf-8')
revision=resultado_df.copy()
revision['lectura_mejora_ocr']=''
revision['lectura_nativa_aceptable']=''
revision['notas_revisor']=''
revision['texto_aprobado_para_rag']=False
revision.to_csv(OUT/'evaluacion_manual.csv',index=False,encoding='utf-8-sig')
print('Comparación visual:',OUT/'revision_ocr.html')

Comparación visual: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa6\revision_ocr.html


## Resultados portables

`etapa6.zip` contiene un resumen, extractos breves y una hoja de evaluación en blanco. Las imágenes y transcripciones completas permanecen en tu computadora. Después de inspeccionar las páginas, puedes completar `evaluacion_manual.csv` (no rellenes las aprobaciones sin revisar la imagen).

In [5]:
extractos=[]
for r in textos_df.itertuples(index=False):
    extractos.append({'mcid':r.mcid,'pagina_pdf':r.pagina_pdf,
        'extracto_nativo':str(r.texto_nativo)[:350],
        'extracto_ocr':str(r.texto_ocr)[:350],
        'caracteres_nativo':len(str(r.texto_nativo)),
        'caracteres_ocr':len(str(r.texto_ocr))})
pd.DataFrame(extractos,columns=['mcid','pagina_pdf','extracto_nativo','extracto_ocr','caracteres_nativo','caracteres_ocr']).to_csv(OUT/'comparacion_extractos.csv',index=False,encoding='utf-8-sig')
reporte={'fecha_utc':datetime.now(timezone.utc).isoformat(),
 'etapa5_sha256':hashlib.sha256(ZIP5.read_bytes()).hexdigest(),
 'etapa4_sha256':hashlib.sha256(ZIP4.read_bytes()).hexdigest(),
 'documentos_total':6,'paginas_objetivo':int(len(seleccion)),
 'paginas_ocr_ejecutado':int((resultado_df.estado=='ocr_ejecutado').sum()),
 'paginas_ocr_sin_texto':int((resultado_df.estado=='ocr_sin_texto').sum()),
 'paginas_error':int((resultado_df.estado=='error_ocr').sum()),
 'motor':'RapidOCR', 'reconocimiento':'PP-OCRv5 latin mobile',
 'ocr_no_validado':True,'paginas_aprobadas_para_rag':0,
 'observaciones':['Confianza del modelo != CER/WER',
   'La legibilidad requiere comparación humana contra imagen original',
   'No corregir automáticamente lenguas indígenas ni ortografía histórica',
   'PDF, imágenes y textos completos solo en disco local',
   'Páginas referidas al PDF, no a numeración impresa']}
(OUT/'informe_etapa6.json').write_text(json.dumps(reporte,ensure_ascii=False,indent=2),encoding='utf-8')
archivo=DATA/'output'/'etapa6.zip'
permitidos=['resumen_ocr_paginas.csv','comparacion_extractos.csv','evaluacion_manual.csv','informe_etapa6.json']
with ZipFile(archivo,'w',ZIP_DEFLATED) as z:
    for f in permitidos: z.write(OUT/f,arcname=f)
print('ARCHIVO PARA COMPARTIR:',archivo)
print(json.dumps(reporte,ensure_ascii=False,indent=2))

ARCHIVO PARA COMPARTIR: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa6.zip
{
  "fecha_utc": "2026-11-09T18:46:15.234886+00:00",
  "etapa5_sha256": "939f82b9cb2b42ad40ffd77b6f5829f620a3f1e3798b1e4ce25d1fd5d2d167d3",
  "etapa4_sha256": "ccb5140dbfd3e682b6cbe77357e389314f9fc84ad983711959c2c96f931b9e1f",
  "documentos_total": 6,
  "paginas_objetivo": 12,
  "paginas_ocr_ejecutado": 12,
  "paginas_ocr_sin_texto": 0,
  "paginas_error": 0,
  "motor": "RapidOCR",
  "reconocimiento": "PP-OCRv5 latin mobile",
  "ocr_no_validado": true,
  "paginas_aprobadas_para_rag": 0,
  "observaciones": [
    "Confianza del modelo != CER/WER",
    "La legibilidad requiere comparación humana contra imagen original",
    "No corregir automáticamente lenguas indígenas ni ortografía histórica",
    "PDF, imágenes y textos completos solo en disco local",
    "Páginas referidas al PDF, no a numeración impresa"
  ]
}
